**Importer les libraries pertinentes**

Source : Prettenhofer, P., Grisel, O., Blondel, M., Amor, A. et Buitinck, L. [Classification of text documents using sparse features](https://scikit-learn.org/stable/auto_examples/text/plot_document_classification_20newsgroups.html#sphx-glr-auto-examples-text-plot-document-classification-20newsgroups-py). *scikit-learn*. 

In [19]:
import multiprocessing
import pandas as pd
from os import listdir
from os import path

from IPython.display import clear_output

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Naive Bayes
from sklearn.naive_bayes import MultinomialNB

# K plus proches voisins (kNN)
from sklearn.neighbors import KNeighborsClassifier

# Machines à vecteurs de support (SVM)
from sklearn import svm

# Régression Logistique
from sklearn.linear_model import LogisticRegression

# Forêt aléatoire 
from sklearn.ensemble import RandomForestClassifier

# Perceptron multicouche
#from sklearn.neural_network import MLPClassifier

**Charger les données**

In [20]:
folder = '../1-data/training_datasets/'
datasets = listdir(folder)
datasets = datasets[8:9]
datasets

['train_dataset_90pc.xlsx']

**Entraîner les modèles et sortir les résultats**

In [21]:
report = []
for file in datasets:
    ratio = file[14:-7]
    
    df = pd.read_excel(path.join(folder, file))
    X = df['text_post'].astype(str)
    y = df['category']

    # 5-fold cross-validation
    stratified_kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    # Valeurs à tester pour le nombre de traits discriminants
    n_features_values = [100, 250, 500, 1000, 2500, 5000, 10000, 15000]

    # Antidictionnaire personnalisé
    with open('./utils/exclusion_v2.stop', encoding='utf-8') as f:
        custom_stopwords = [x.lower().strip('\n') for x in f.readlines()]
        custom_stopwords += list(ENGLISH_STOP_WORDS)

    # Algorithmes à tester
    algorithms = {
        'Naive Bayes' : MultinomialNB(), 
        'kNN (k=1)' : KNeighborsClassifier(n_neighbors=1),
        'kNN (k=2)' : KNeighborsClassifier(n_neighbors=2),
        'kNN (k=3)' : KNeighborsClassifier(n_neighbors=3),
        'Support Vector Machines (SVM)' :  svm.SVC(kernel="linear", C=1.0),
        'Logistic Regression': LogisticRegression(C=5, max_iter=1000),
        'Random Forest': RandomForestClassifier(),
        #'Multilayered Perceptron': MLPClassifier(hidden_layer_sizes=(100,), max_iter=500)
    }

    for n_features in n_features_values:
            tfidf_bow_vectorizer = TfidfVectorizer(stop_words=custom_stopwords, max_features=n_features)
            
            for algorithm in algorithms.keys():
                pipeline_algo = Pipeline([
                    ('tfidf', tfidf_bow_vectorizer),
                    ('classifier', algorithms[algorithm])
                ])

                # Validation croisée 
                scores = cross_val_score(pipeline_algo, X, y, cv=stratified_kfold, scoring='accuracy')
                predictions = cross_val_predict(pipeline_algo, X, y, cv=stratified_kfold)

                # Performances (rappel, précision, score F)
                results = {
                    '% Incels' : int(ratio),
                    'Algorithme' : algorithm,
                    'N traits discr.' : n_features,
                    'accuracy': scores.mean()       
                }
                results.update(classification_report(y, predictions, output_dict=True)['macro avg'])
                report.append(results)
                
                print(algorithm, f' | {ratio}% Incels | ', f'{n_features} traits discr.\n', classification_report(y, predictions))
                clear_output(wait=True)

Random Forest  | 90% Incels |  15000 traits discr.
               precision    recall  f1-score   support

       incel       0.91      0.98      0.94     45000
     neutral       0.40      0.09      0.15      4999

    accuracy                           0.90     49999
   macro avg       0.65      0.54      0.55     49999
weighted avg       0.86      0.90      0.87     49999



**Résultats**

In [22]:
# N.B. : Support = Nombre d'occurrence dans chaque classe
report = pd.DataFrame(report)
report['nb_posts_incels'] = (report['% Incels'].apply(lambda x: x/100) * report['support']).astype(int)
report['nb_posts_neutral'] = (report['% Incels'].apply(lambda x: 1-(x/100)) * report['support']).astype(int)

report = report.rename(columns={'support':'nb_posts_total'})
report['nb_posts_total'] = report['nb_posts_total'].astype(int)

# Réorganiser l'ordre des colonnes
report = report[['nb_posts_total', 'nb_posts_incels', 'nb_posts_neutral', '% Incels', 'Algorithme', 
                 'N traits discr.', 'accuracy', 'precision', 'recall','f1-score']]

report

,nb_posts_total,nb_posts_incels,nb_posts_neutral,% Incels,Algorithme,N traits discr.,accuracy,precision,recall,f1-score
0,49999,44999,4999,90,Naive Bayes,100,0.9000,0.4500,0.5000,0.4737
1,49999,44999,4999,90,kNN (k=1),100,0.8854,0.4999,0.5000,0.4849
2,49999,44999,4999,90,kNN (k=2),100,0.8986,0.5285,0.5006,0.4764
3,49999,44999,4999,90,kNN (k=3),100,0.8960,0.5158,0.5010,0.4794
4,49999,44999,4999,90,Support Vector Machines (SVM),100,0.9000,0.4500,0.5000,0.4737
5,49999,44999,4999,90,Logistic Regression,100,0.9000,0.4500,0.5000,0.4737
6,49999,44999,4999,90,Random Forest,100,0.8968,0.5274,0.5013,0.4793
7,49999,44999,4999,90,Naive Bayes,250,0.9000,0.4500,0.5000,0.4737
8,49999,44999,4999,90,kNN (k=1),250,0.8784,0.5161,0.5051,0.4972
9,49999,44999,4999,90,kNN (k=2),250,0.8982,0.5766,0.5032,0.4822


**Exporter les résultats de l'apprentissage**

In [23]:
report.sort_values(by='f1-score', ascending=False).to_csv(
    f'../3-results/results_training_90pc.csv', index=False
)

**Tester le système retenu sur de nouvelles données**

*On reproduit l'apprentissage pour le système retenu uniquement*

In [6]:
file_train = '../1-data/training_datasets/train_dataset_50pc.xlsx'

df_train = pd.read_excel(file_train)
df_train

,id_post,date_post,subreddit,author,text_post,category
0,g5cmft2,2020,ForeverAloneDating,luxiuo,Happy cake day my dude! Hope you get at least ...,incel
1,i3govkj,2022,ForeverAlone,Cassofalltrades,"Yeah, I always fantasized falling in love with...",incel
2,dkbpw8p,2017,Incels,shingis231,bro they're incels in denial. no fucking way a...,incel
3,dih6nor,2017,Incels,SmokeyMcBlunt,"lmao @ ""Posting on reddit""\n\nI wonder if thes...",incel
4,d8a5g44,2016,Incels,not_much_left,This is not true. Social research is fine so ...,incel
...,...,...,...,...,...,...
49995,esfzq10,2019,veloster,trav1th3rabb1,Yeah and what can do with this too?,neutral
49996,dzwtdlf,2018,gaming,jacksonh_56,I only have the snes classic but I want to own...,neutral
49997,hewyzgg,2021,insaneparents,Sheepgomoo123,What’s with these people thinking the “super s...,neutral
49998,esg0204,2019,hockey,Skylightt,I don't know about likely but Vatanen is defin...,neutral


In [7]:
# Système retenu : Naive Bayes ; 40% de données Incels ; 5 000 traits discriminants
X_train = df_train['text_post'].astype(str)
y_train = df_train['category']

ngram_values = (1, 2)

# Antidictionnaire personnalisé
with open('./utils/exclusion_v2.stop', encoding='utf-8') as f:
    custom_stopwords = [x.lower().strip('\n') for x in f.readlines()]
    custom_stopwords += list(ENGLISH_STOP_WORDS)

pipeline_nb = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words=custom_stopwords, ngram_range=ngram_values, max_features=15000)),
    ('nb', MultinomialNB())
])

pipeline_nb.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=15000, ngram_range=(1, 2),
                                 stop_words=['\ufeff&gt', 'a', 'â', 'ä', 'ã',
                                             'å', 'ªã', 'about', 'actually',
                                             'æ', 'again', 'against', "ain't",
                                             'all', 'allow', 'allows', 'almost',
                                             'alone', 'along', 'already',
                                             'also', 'although', 'always', 'am',
                                             'among', 'amongst', 'amp', 'an',
                                             'and', 'another', ...])),
                ('nb', MultinomialNB())])

In [8]:
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
nb_predictions = cross_val_predict(pipeline_nb, X_train, y_train, cv=stratified_kfold)

# Performances en phase d'apprentissage du modèle retenu (rappel, précision, score F)
# Calculer et afficher les performances
accuracy_nb = accuracy_score(y_train, nb_predictions)
precision_nb = precision_score(y_train, nb_predictions, average='macro')
recall_nb = recall_score(y_train, nb_predictions, average='macro')
f1_nb = f1_score(y_train, nb_predictions, average='macro')

df = [
    {
        'Phase': 'Apprentissage',
        'Précision': precision_nb,
        'Rappel': recall_nb,
        'Mesure F': f1_nb,
        'Exactitude': accuracy_nb

    }
]

print(classification_report(y_train, nb_predictions))

              precision    recall  f1-score   support

       incel       0.72      0.81      0.76     25000
     neutral       0.78      0.68      0.73     25000

    accuracy                           0.75     50000
   macro avg       0.75      0.75      0.75     50000
weighted avg       0.75      0.75      0.75     50000



*On teste sur de nouvelles données*

In [9]:
file_test = '../1-data/test_dataset_10pc.xlsx'

df_test = pd.read_excel(file_test)
df_test

,date_post,id_post,author,subreddit,text_post,category
0,2021,hewyqir,KallystoNyx,FreeKarma4U,Bang!,neutral
1,2022,hv2cm15,Hopeful-Talk-1556,canada,I mean this is why you often get downvoted in ...,neutral
2,2021,hewyfkk,naughtyrpfan,HentaiAndRoleplayy,You playing m or f?,neutral
3,2016,d1l6201,greigmusic,IAmA,"Lol, google black pudding and your view will c...",neutral
4,2020,fzxnd3h,NorCalifornioAH,MapPorn,That's more a matter of Combined Statistical A...,neutral
...,...,...,...,...,...,...
9995,2021,hgdqoim,throwamay555,ForeverAlone,who said you were ugly?,incel
9996,2021,hmqkhx0,Admirable_Elk_965,ForeverAlone,Yeah I see this too when I walk out of class. ...,incel
9997,2022,iw0zptn,Resident_Setting3884,ForeverAlone,"this is exactly how i feel, im a cab driver, t...",incel
9998,2017,dfiq9mw,CALLMEMANTEE,Incels,I mean I'm not ok with it because its depressi...,incel


*Vérifier qu'aucune donnée test n'était contenue dans les données d'appentissage*

In [10]:
validate = pd.concat([df_train, df_test])

validate[validate.duplicated(subset='id_post')]

,id_post,date_post,subreddit,author,text_post,category


*Appliquer le classifieur aux nouvelles données*

In [11]:
X_test = df_test['text_post'].astype(str)
y_test = df_test['category']

y_pred_nb = pipeline_nb.predict(X_test)

In [12]:
# Calculer et afficher les performances
accuracy_nb = accuracy_score(y_test, y_pred_nb)
precision_nb = precision_score(y_test, y_pred_nb, average='macro')
recall_nb = recall_score(y_test, y_pred_nb, average='macro')
f1_nb = f1_score(y_test, y_pred_nb, average='macro')

df.append(
    {
        'Phase': 'Test',
        'Précision': precision_nb,
        'Rappel': recall_nb,
        'Mesure F': f1_nb,
        'Exactitude': accuracy_nb

    }
)

pd.set_option("display.precision", 4)
df = pd.DataFrame(df)
df.to_csv('../3-results/results_test.csv', index=False)
df

,Phase,Précision,Rappel,Mesure F,Exactitude
0,Apprentissage,0.7509,0.7468,0.7458,0.7468
1,Test,0.5976,0.7490,0.5804,0.7042


In [13]:
accuracy_nb

0.7042

In [14]:
precision_nb

0.5976497138745733

In [15]:
recall_nb

0.749

In [16]:
f1_nb

0.5803861060647741

In [17]:
print(classification_report(y_test, y_pred_nb))

              precision    recall  f1-score   support

       incel       0.23      0.81      0.35      1000
     neutral       0.97      0.69      0.81      9000

    accuracy                           0.70     10000
   macro avg       0.60      0.75      0.58     10000
weighted avg       0.90      0.70      0.76     10000



In [18]:
import pandas as pd
results = pd.read_csv('../3-results/results_training.csv')
results.sort_values(by='f_score', ascending=False)

KeyError: 'f_score'